# Running the benchmark with a custom pytorch-like model

### Step 1: Import libraries

Import PyTorch components and nirsatlas deep learning utilities.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.pipeline import Pipeline
from fnirsatlas.dl.DLModelBase import DLModelBase
from fnirsatlas.feature_extraction import StandardScalerND, ChannelSplitter
from fnirsatlas.classification import run_benchmark

### Step 2: Define CNN model

Create a CNN extending **DLModelBase** with sklearn-like interface:
- 1D convolutions for temporal then spatial filtering  
- ReLU activations and dropout for regularization
- Linear layer for final classification

In [2]:
class SimpleNet(DLModelBase):
    def __init__(self, T, n_chs, num_classes, p_dropout=0.5, **kwargs):
        super().__init__(**kwargs)
        self.T = T # All model parameters need to be accessible as attributes
        self.n_chs = n_chs
        self.num_classes = num_classes
        self.p_dropout = p_dropout
        self.layers = nn.Sequential(
            nn.Conv2d(2, 4, kernel_size=(1, T)),
            nn.ReLU(),
            nn.Dropout(self.p_dropout),
            nn.Conv2d(4, 8, kernel_size=(n_chs, 1)),
            nn.ReLU(),
            nn.Dropout(self.p_dropout),
            nn.Flatten(),
            nn.Linear(8, num_classes),
        )

    def forward(self, x):
        return self.layers(x)


### Step 3: Create pipeline factory

Build pipeline with data preprocessing and model:
1. **ChannelSplitter**: Reshapes data from (n_samples, n_total_chs, T) to (n_samples, n_ch_types, n_chs_per_type, T)
2. **StandardScalerND**: Z-normalizes features across samples  
3. **SimpleNet**: Custom CNN classifier

In [3]:
def get_clf(**kwargs):
    random_state = kwargs["random_state"]
    num_classes = np.unique(kwargs["y"]).shape[0]
    ch_names = kwargs["info"]["ch_names"]
    T = kwargs["X"].shape[-1]
    n_chs = len(ch_names) // 2
    torch.manual_seed(random_state)
    return Pipeline(
        [
            ("splitter", ChannelSplitter(ch_names)),
            ("scaler", StandardScalerND(dim=(0, 2, 3))),
            ("clf", SimpleNet(T, n_chs, num_classes, normalize=False)),
        ]
    )

### Step 4: Run benchmark

Execute both cross- and within-subject classification with dropout hyperparameter tuning using nested cross-validation (seeds × 5-fold outer CV, 4-fold inner CV by default).

In [4]:
res = run_benchmark(
    get_clf=get_clf,
    suite="ma",
    classification_setting="both",
    seeds=[42], # 1 seed used for quick testing
    param_grid={
        "clf__p_dropout": [0.1, 0.5, 0.9],
    },
)

Classifying datasets: 100%|██████████| 4/4 [01:16<00:00, 19.16s/it, shin2019__ma]


Summary:
dataset          task                           #classes  #epochs   Cross%    Within%   Cross κ    Within κ
--------------------------------------------------------------------------------------------------------------
Bnci             BL|MA                                 2      348  88.8±7.2    93.4±5.1  77.5±14.4    86.8±10.1
Shin2016 (ma)    BL|MA                                 2     1740  64.3±1.6   70.7±11.3   28.6±3.3    41.5±22.5
Shin2018b        BL|MA                                 2      720  62.8±6.4    64.3±9.8  25.6±12.8    28.6±19.7
Shin2019 (ma)    BL|MA                                 2     1080  65.6±4.8    67.4±9.4   31.2±9.5    34.8±18.9
--------------------------------------------------------------------------------------------------------------
MEAN                                                               70.4±5.0    74.0±8.9  40.7±10.0    47.9±17.8
